# रामचरितमानस पाठ · open-source verse audio
Run the cells **top to bottom** with the ▶ button.

Before you start: **Runtime → Change runtime type → T4 GPU → Save**.

Use **one engine per session** (Parler *or* IndicF5). To switch, choose **Runtime → Disconnect and delete runtime** and start again from the top.

## 1 · Check the GPU and get the app

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/shubhampost4u/ramcharitmanas-path
%cd ramcharitmanas-path
!git log -1 --format='Repo ready at %h: %s'

## 2 · Sign in to Hugging Face
You need an account that has accepted the terms on the model page:
- Parler: https://huggingface.co/ai4bharat/indic-parler-tts
- IndicF5: https://huggingface.co/ai4bharat/IndicF5

Create a token at https://huggingface.co/settings/tokens (type **Read**) and paste it when asked.

In [ ]:
from huggingface_hub import login
login()

## 3A · Engine: Indic Parler-TTS (no recording needed)
Install (about 2 minutes).

In [ ]:
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile

Make 5 test verses, then listen. Change `VOICE` to try another speaker (Rohit, Aman, Divya, Rani) or a different pace, and run again.

In [ ]:
VOICE = "Rohit speaks slowly and calmly in a deep, warm voice, reciting devotional verse with gentle rhythm and clear pauses at each comma and full stop. The recording is very high quality with no background noise."
!rm -rf audio-test/parler
!python tools/oss_tts.py --engine parler --test --voice "$VOICE"

from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('audio-test/parler/*.mp3')):
    print(f); display(Audio(f))

Happy with the test? Voice the whole Sundar Kand (about 60-90 minutes; keep this tab open). If it stops, just run the cell again: finished verses are skipped.

Remove `--as-default` if you want to keep the Sarvam voice as what the app plays.

In [ ]:
!python tools/oss_tts.py --engine parler --voice "$VOICE" --as-default

## 3B · Engine: IndicF5 (copies your recitation style)
Install (about 3 minutes).

In [ ]:
!pip -q install git+https://github.com/ai4bharat/IndicF5.git soundfile

Upload your reference recording: one chaupai, 8-15 seconds, recited the way you want every verse to sound, in a quiet room. Then type **exactly** the words you recited.

In [ ]:
from google.colab import files
up = files.upload()
REF = list(up.keys())[0]
REF_TEXT = "जामवंत के बचन सुहाए। सुनि हनुमंत हृदय अति भाए॥"   # edit to match your recording exactly
print('Reference:', REF)

In [ ]:
!rm -rf audio-test/indicf5
!python tools/oss_tts.py --engine indicf5 --test --ref "$REF" --ref-text "$REF_TEXT"

from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('audio-test/indicf5/*.mp3')):
    print(f); display(Audio(f))

Whole Kand with IndicF5 (about 60-90 minutes). Run again if it stops.

In [ ]:
!python tools/oss_tts.py --engine indicf5 --ref "$REF" --ref-text "$REF_TEXT" --as-default

## 4 · Save to GitHub
Create a token at https://github.com/settings/personal-access-tokens/new → **Only select repositories** → `ramcharitmanas-path` → Permissions → **Contents: Read and write**. You can reuse your podcast token if it already has this repo.

In [ ]:
import os, getpass
os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub token: ')
!python tools/oss_tts.py --push-only